# ML workflow for PM2.5 prediction

This notebook documents a complete and honest modeling workflow for PM2.5 prediction:

- loading and preprocessing the dataset,
- checking data quality,
- testing feature engineering without leakage,
- comparing baseline models with cross-validation,
- tuning Random Forest and Gradient Boosting,
- evaluating the final models on a hold-out test set,
- generating a real-vs-predicted comparison curve.

The markdown comments and printed explanations are written to be reusable in a report.

In [1]:
import sys
import json
import warnings
from pathlib import Path

import altair as alt
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")
alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.preprocess import build_preprocessor_from_settings, load_raw_dataframe, preprocess_dataframe
from src.utils.config import get_nested, load_settings, resolve_path

def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

def explain(message):
    print("\n" + message + "\n")

def build_pipeline(estimator):
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", estimator),
        ]
    )


## 1. Load the dataset and check data quality

This first block verifies whether the dataset is usable for regression before starting any modeling work. We focus on duplicates, missing values, and the completeness of the variables actually used by the models.

In [2]:
settings = load_settings(PROJECT_ROOT / "config/settings.yaml")
preprocessor = build_preprocessor_from_settings(settings)
raw_df = load_raw_dataframe(PROJECT_ROOT / "data/raw/air_quality_combined_source.csv")
df = preprocess_dataframe(raw_df, preprocessor).copy()

raw_features = get_nested(settings, "ml", "feature_columns", default=[])
target_column = get_nested(settings, "ml", "target_column", default="pm25")
model_df_raw = df[raw_features + [target_column]].copy()

quality_summary = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "rows_after_preprocessing",
            "removed_duplicates",
            "missing_target_pm25",
            "missing_values_in_model_features",
            "missing_raw_date",
        ],
        "value": [
            len(raw_df),
            len(df),
            len(raw_df) - len(df),
            int(model_df_raw[target_column].isna().sum()),
            int(model_df_raw[raw_features].isna().sum().sum()),
            int(raw_df["date"].isna().sum()) if "date" in raw_df.columns else 0,
        ],
    }
)

display(quality_summary)
display(model_df_raw.describe().T)

explain(
    f"Interpretation: the dataset contains {len(df)} usable rows after preprocessing. "
    f"For the modeling variables, missing values are effectively zero and the target {target_column} is complete. "
    f"This means we can train regression models without heavy imputation or row deletion. "
    f"The raw date column is mostly empty, so it is not retained as a reliable predictor in this first modeling pass."
)


,metric,value
0,raw_rows,915
1,rows_after_preprocessing,915
2,removed_duplicates,0
3,missing_target_pm25,0
4,missing_values_in_model_features,0
5,missing_raw_date,910


,count,mean,std,min,25%,50%,75%,max
pm10,915.000,107.455,51.218,20.044,64.258,105.689,151.411,199.920
no2,915.000,62.692,33.492,5.370,33.987,63.620,91.434,119.810
so2,915.000,39.662,22.871,1.018,19.332,39.423,60.023,79.913
co,915.000,4.878,2.912,0.113,2.268,4.740,7.391,9.994
o3,915.000,80.820,40.311,10.057,47.699,79.544,115.003,149.889
temperature,915.000,24.798,8.697,10.001,17.083,24.999,31.846,39.992
humidity,915.000,55.919,20.852,20.532,37.955,55.311,72.644,94.831
wind_speed,915.000,4.143,2.201,0.502,2.115,4.155,6.093,7.996
pm25,915.000,94.124,48.084,10.023,53.775,92.695,134.369,179.795



Interpretation: the dataset contains 915 usable rows after preprocessing. For the modeling variables, missing values are effectively zero and the target pm25 is complete. This means we can train regression models without heavy imputation or row deletion. The raw date column is mostly empty, so it is not retained as a reliable predictor in this first modeling pass.



## 2. Test feature engineering without leakage

We create interaction and non-linear terms **only from predictors**. The target `pm25` is never used to build a feature, because that would create data leakage and produce misleadingly optimistic results.

Before keeping the engineered variables, we check whether they improve cross-validated performance on a representative non-linear model.

In [3]:
model_df_engineered = model_df_raw.copy()
model_df_engineered["no2_o3_interaction"] = model_df_engineered["no2"] * model_df_engineered["o3"]
model_df_engineered["temp_humidity_interaction"] = model_df_engineered["temperature"] * model_df_engineered["humidity"]
model_df_engineered["co_so2_interaction"] = model_df_engineered["co"] * model_df_engineered["so2"]
model_df_engineered["wind_pm10_interaction"] = model_df_engineered["wind_speed"] * model_df_engineered["pm10"]
model_df_engineered["pollution_sum"] = model_df_engineered[["pm10", "no2", "so2", "co", "o3"]].sum(axis=1)
model_df_engineered["weather_stress"] = model_df_engineered["temperature"] / (model_df_engineered["humidity"] + 1e-6)
model_df_engineered["temperature_sq"] = model_df_engineered["temperature"] ** 2
model_df_engineered["humidity_sq"] = model_df_engineered["humidity"] ** 2

engineered_features = [
    "no2_o3_interaction",
    "temp_humidity_interaction",
    "co_so2_interaction",
    "wind_pm10_interaction",
    "pollution_sum",
    "weather_stress",
    "temperature_sq",
    "humidity_sq",
]

feature_set_results = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

feature_sets = {
    "raw": (model_df_raw, raw_features),
    "engineered": (model_df_engineered, raw_features + engineered_features),
}

for feature_set_name, (feature_frame, feature_list) in feature_sets.items():
    X_current = feature_frame[feature_list]
    y_current = feature_frame[target_column]
    pipeline = build_pipeline(GradientBoostingRegressor(random_state=42))
    scores = cross_validate(pipeline, X_current, y_current, cv=cv, scoring=scoring, n_jobs=1)
    feature_set_results.append(
        {
            "feature_set": feature_set_name,
            "cv_rmse": -scores["test_rmse"].mean(),
            "cv_mae": -scores["test_mae"].mean(),
            "cv_r2": scores["test_r2"].mean(),
        }
    )

feature_set_results = pd.DataFrame(feature_set_results).sort_values("cv_rmse").reset_index(drop=True)
selected_feature_set_name = feature_set_results.iloc[0]["feature_set"]
selected_model_df, selected_features = feature_sets[selected_feature_set_name]

display(feature_set_results)

if selected_feature_set_name == "engineered":
    explain(
        "Interpretation: the engineered features improve cross-validated RMSE on the validation folds, "
        "so they are kept for the rest of the notebook."
    )
else:
    explain(
        "Interpretation: the engineered variables do not improve the cross-validated error on this dataset. "
        "To stay methodologically honest, the rest of the workflow keeps the raw predictors only. "
        "This is an important result: feature engineering must be validated, not assumed to help."
    )


,feature_set,cv_rmse,cv_mae,cv_r2
0,raw,48.621,40.965,-0.024
1,engineered,49.525,41.751,-0.063



Interpretation: the engineered variables do not improve the cross-validated error on this dataset. To stay methodologically honest, the rest of the workflow keeps the raw predictors only. This is an important result: feature engineering must be validated, not assumed to help.



## 3. Compare baseline models with cross-validation

Now we compare three baseline models on the selected feature set:

- Linear Regression,
- Random Forest,
- Gradient Boosting.

The comparison uses 5-fold cross-validation and focuses on `RMSE`, `MAE`, and `R2`.

In [4]:
X = selected_model_df[selected_features]
y = selected_model_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

baseline_rows = []
for model_name, estimator in baseline_models.items():
    pipeline = build_pipeline(estimator)
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    baseline_rows.append(
        {
            "model": model_name,
            "cv_rmse_mean": -scores["test_rmse"].mean(),
            "cv_mae_mean": -scores["test_mae"].mean(),
            "cv_r2_mean": scores["test_r2"].mean(),
        }
    )

baseline_results = pd.DataFrame(baseline_rows).sort_values("cv_rmse_mean").reset_index(drop=True)
display(baseline_results)

best_baseline = baseline_results.iloc[0]
worst_baseline = baseline_results.iloc[-1]
explain(
    f"Interpretation: among the untuned baselines, {best_baseline['model']} obtains the lowest average RMSE in cross-validation. "
    f"However, the R2 values remain weak or negative, which means the predictive signal is limited. "
    f"The gap between the best and worst baseline is also small, so we should be careful not to over-claim model superiority at this stage."
)


,model,cv_rmse_mean,cv_mae_mean,cv_r2_mean
0,Linear Regression,48.691,41.604,-0.042
1,Random Forest,48.717,41.527,-0.044
2,Gradient Boosting,49.628,41.780,-0.084



Interpretation: among the untuned baselines, Linear Regression obtains the lowest average RMSE in cross-validation. However, the R2 values remain weak or negative, which means the predictive signal is limited. The gap between the best and worst baseline is also small, so we should be careful not to over-claim model superiority at this stage.



## 4. Tune Random Forest and Gradient Boosting

We keep Linear Regression as a simple benchmark and tune the two non-linear models with a **light RandomizedSearchCV**. This keeps execution time acceptable while still testing whether hyperparameter optimization brings a measurable gain.

In [5]:
rf_search = RandomizedSearchCV(
    estimator=build_pipeline(RandomForestRegressor(random_state=42, n_jobs=1)),
    param_distributions={
        "model__n_estimators": [150, 250, 350],
        "model__max_depth": [None, 8, 12, 16],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [1.0, "sqrt"],
    },
    n_iter=4,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    random_state=42,
)

gb_search = RandomizedSearchCV(
    estimator=build_pipeline(GradientBoostingRegressor(random_state=42)),
    param_distributions={
        "model__n_estimators": [150, 250, 350],
        "model__learning_rate": [0.03, 0.05, 0.08],
        "model__max_depth": [2, 3, 4],
        "model__subsample": [0.8, 1.0],
    },
    n_iter=4,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    random_state=42,
)

rf_search.fit(X_train, y_train)
gb_search.fit(X_train, y_train)

tuning_results = pd.DataFrame(
    [
        {
            "model": "Random Forest tuned",
            "best_cv_rmse": -rf_search.best_score_,
            "best_params": json.dumps(rf_search.best_params_),
        },
        {
            "model": "Gradient Boosting tuned",
            "best_cv_rmse": -gb_search.best_score_,
            "best_params": json.dumps(gb_search.best_params_),
        },
    ]
).sort_values("best_cv_rmse").reset_index(drop=True)

display(tuning_results)
explain(
    "Interpretation: tuning helps us verify whether non-linear models can extract more signal from the available predictors. "
    "If the tuned CV RMSE stays close to the baseline RMSE, the main limitation is probably the dataset itself rather than the choice of hyperparameters."
)


,model,best_cv_rmse,best_params
0,Random Forest tuned,48.392,"{""model__n_estimators"": 150, ""model__min_sampl..."
1,Gradient Boosting tuned,48.590,"{""model__subsample"": 1.0, ""model__n_estimators..."



Interpretation: tuning helps us verify whether non-linear models can extract more signal from the available predictors. If the tuned CV RMSE stays close to the baseline RMSE, the main limitation is probably the dataset itself rather than the choice of hyperparameters.



## 5. Final evaluation on the hold-out test set

This is the most important comparison in the notebook. The test set is not used during training or hyperparameter search, so these metrics are the ones to keep for an honest conclusion.

In [6]:
final_models = {
    "Linear Regression": build_pipeline(LinearRegression()),
    "Random Forest tuned": rf_search.best_estimator_,
    "Gradient Boosting tuned": gb_search.best_estimator_,
}

prediction_store = {}
final_rows = []
for model_name, pipeline in final_models.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    prediction_store[model_name] = predictions
    final_rows.append(
        {
            "model": model_name,
            "test_rmse": rmse(y_test, predictions),
            "test_mae": float(mean_absolute_error(y_test, predictions)),
            "test_r2": float(r2_score(y_test, predictions)),
        }
    )

final_results = pd.DataFrame(final_rows).sort_values("test_rmse").reset_index(drop=True)
best_final_model_name = final_results.iloc[0]["model"]
display(final_results)

summary_payload = {
    "selected_feature_set": selected_feature_set_name,
    "baseline_results": baseline_results.to_dict(orient="records"),
    "tuning_results": tuning_results.to_dict(orient="records"),
    "final_results": final_results.to_dict(orient="records"),
}
summary_path = resolve_path("models/notebook_model_selection_summary.json")
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

best_row = final_results.iloc[0]
explain(
    f"Interpretation: on the unseen test set, the best model is {best_row['model']} with RMSE={best_row['test_rmse']:.3f}, "
    f"MAE={best_row['test_mae']:.3f}, and R2={best_row['test_r2']:.3f}. "
    "These values should be quoted before any metric computed on the full dataset, because they are less optimistic and methodologically stronger."
)


,model,test_rmse,test_mae,test_r2
0,Linear Regression,49.026,41.869,-0.008
1,Gradient Boosting tuned,49.224,41.916,-0.017
2,Random Forest tuned,49.685,42.631,-0.036



Interpretation: on the unseen test set, the best model is Linear Regression with RMSE=49.026, MAE=41.869, and R2=-0.008. These values should be quoted before any metric computed on the full dataset, because they are less optimistic and methodologically stronger.



## 6. Plot actual values vs predictions

The chart below compares the true PM2.5 values against the predictions of the final candidate models on the hold-out test set only. This is the fairest visual comparison in the notebook.

The cell also exports:

- `models/notebook_actual_vs_predicted.csv`
- `models/notebook_actual_vs_predicted.html`

In [7]:
prediction_frame = pd.DataFrame(
    {
        "sample_index": list(range(len(y_test))),
        "actual_pm25": y_test.reset_index(drop=True),
    }
)

for model_name, predictions in prediction_store.items():
    safe_name = model_name.lower().replace(" ", "_")
    prediction_frame[f"prediction_{safe_name}"] = predictions

long_plot_df = prediction_frame.melt(
    id_vars="sample_index",
    value_vars=[col for col in prediction_frame.columns if col != "sample_index"],
    var_name="series",
    value_name="value",
)

label_map = {
    "actual_pm25": "Actual PM2.5",
    "prediction_linear_regression": "Prediction - Linear Regression",
    "prediction_random_forest_tuned": "Prediction - Random Forest tuned",
    "prediction_gradient_boosting_tuned": "Prediction - Gradient Boosting tuned",
}
long_plot_df["series"] = long_plot_df["series"].map(label_map).fillna(long_plot_df["series"])

chart = (
    alt.Chart(long_plot_df)
    .mark_line(strokeWidth=2.5)
    .encode(
        x=alt.X("sample_index:Q", title="Test sample"),
        y=alt.Y("value:Q", title="PM2.5"),
        color=alt.Color("series:N", title="Series"),
        tooltip=[
            alt.Tooltip("sample_index:Q", title="Test sample"),
            alt.Tooltip("series:N", title="Series"),
            alt.Tooltip("value:Q", title="Value", format=".2f"),
        ],
    )
    .properties(height=380, title="Actual PM2.5 vs model predictions on the hold-out test set")
    .interactive()
)

plot_path = resolve_path("models/notebook_actual_vs_predicted.html")
csv_path = resolve_path("models/notebook_actual_vs_predicted.csv")
plot_path.parent.mkdir(parents=True, exist_ok=True)
prediction_frame.to_csv(csv_path, index=False)
chart.save(plot_path)

display(chart)
explain(
    f"Interpretation: this curve makes model behavior easy to inspect. If the prediction line follows the actual line closely, "
    f"the model captures the changes in PM2.5 reasonably well. The best model retained in this notebook is {best_final_model_name}. "
    f"The exported CSV and HTML files can be reused directly in a report or presentation."
)


alt.Chart(...)


Interpretation: this curve makes model behavior easy to inspect. If the prediction line follows the actual line closely, the model captures the changes in PM2.5 reasonably well. The best model retained in this notebook is Linear Regression. The exported CSV and HTML files can be reused directly in a report or presentation.



## 7. Final conclusion

The final printed text below is intentionally written in a report-friendly style.

In [8]:
best_model_row = final_results.iloc[0]

if best_model_row["test_r2"] >= 0.5:
    conclusion = (
        f"Conclusion: {best_model_row['model']} is the most credible final candidate in this notebook. "
        f"On the hold-out test set it reaches RMSE={best_model_row['test_rmse']:.3f}, "
        f"MAE={best_model_row['test_mae']:.3f}, and R2={best_model_row['test_r2']:.3f}. "
        "The combination of preprocessing, feature selection, cross-validation, and tuning produced a meaningful gain."
    )
else:
    conclusion = (
        f"Conclusion: the best model in this notebook is {best_model_row['model']}, "
        f"but the hold-out performance remains limited (RMSE={best_model_row['test_rmse']:.3f}, "
        f"MAE={best_model_row['test_mae']:.3f}, R2={best_model_row['test_r2']:.3f}). "
        "This suggests that the main bottleneck is the available signal in the dataset rather than the absence of modeling effort. "
        "In other words, the workflow is methodologically stronger now, but better prediction will likely require richer or more reliable data."
    )

print(conclusion)
print("\nSaved artifacts:")
print(f"- {summary_path}")
print(f"- {plot_path}")
print(f"- {csv_path}")


Conclusion: the best model in this notebook is Linear Regression, but the hold-out performance remains limited (RMSE=49.026, MAE=41.869, R2=-0.008). This suggests that the main bottleneck is the available signal in the dataset rather than the absence of modeling effort. In other words, the workflow is methodologically stronger now, but better prediction will likely require richer or more reliable data.

Saved artifacts:
- C:\Users\DELL\Documents\pacte\Smart-City---Dashboard-de-surveillance-de-la-qualite-de-l-air\models\notebook_model_selection_summary.json
- C:\Users\DELL\Documents\pacte\Smart-City---Dashboard-de-surveillance-de-la-qualite-de-l-air\models\notebook_actual_vs_predicted.html
- C:\Users\DELL\Documents\pacte\Smart-City---Dashboard-de-surveillance-de-la-qualite-de-l-air\models\notebook_actual_vs_predicted.csv
